# 01 - Data Preparation
**Project:** Credit Card Fraud Detection (ULB benchmark, OpenML ID 1597)
**Syllabus link:** Introduction to predictive analytics, supervised learning, data preprocessing

**Problem type:** supervised binary classification (`Class` = 1 fraud, 0 legitimate), labelled data.

Steps: load, inspect, missing values, duplicates, outliers, scaling demo, feature engineering, save cleaned data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, StandardScaler

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 40)

RAW_PATH = "../data/raw/creditcard.csv"
PROC_DIR = "../data/processed"
FIG_DIR = "../reports/figures"
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

## 1. Load and inspect

In [ ]:
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

**Dataset note:** `Time` = seconds since the first transaction (2 days of data). `V1-V28` are PCA components (anonymised for confidentiality). `Amount` = transaction amount. `Class` = target.

## 2. Missing values

In [ ]:
missing = df.isnull().sum()
print("Total missing values:", missing.sum())
missing[missing > 0]

The dataset has no missing values, so no imputation is needed. This is stated explicitly in the report.

## 3. Class distribution (raw)

In [ ]:
counts = df["Class"].value_counts()
print(counts)
print((counts / len(df) * 100).round(3))

## 4. Duplicates

In [ ]:
n_dup = df.duplicated().sum()
print("Duplicate rows:", n_dup)
print("Fraud rows among duplicates:", df[df.duplicated()]["Class"].sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)
print(df["Class"].value_counts())

Exact duplicate rows add no information and can leak between train and test splits, so they are removed.

## 5. Outliers in `Amount` (IQR method)

In [ ]:
Q1, Q3 = df["Amount"].quantile([0.25, 0.75])
IQR = Q3 - Q1
low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
is_out = (df["Amount"] < low) | (df["Amount"] > high)

print(f"IQR bounds: [{low:.2f}, {high:.2f}]")
print("Outlier rows:", is_out.sum(), f"({is_out.mean()*100:.1f}%)")
print("Fraud rate among outliers:", round(df.loc[is_out, "Class"].mean() * 100, 3), "%")
print("Fraud rate among non-outliers:", round(df.loc[~is_out, "Class"].mean() * 100, 3), "%")
print("Skewness of Amount:", round(df["Amount"].skew(), 2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.boxplot(x="Class", y="Amount", data=df, ax=ax[0])
ax[0].set_title("Amount by class (with outliers)")
sns.boxplot(x="Class", y="Amount", data=df, showfliers=False, ax=ax[1])
ax[1].set_title("Amount by class (outliers hidden)")
ax[2].hist(np.log1p(df["Amount"]), bins=60)
ax[2].set_title("log(1 + Amount)")
plt.tight_layout()
savefig("01_amount_outliers")
plt.show()

**Decision:** outliers are **kept**. High-amount transactions are legitimate behaviour for the dataset, and removing outliers could delete real fraud cases (only ~473 frauds exist). Instead, a robust scaler is used to limit their influence.

## 6. Scaling (demonstration)

In [ ]:
cols = ["Time", "Amount"]
std = pd.DataFrame(StandardScaler().fit_transform(df[cols]), columns=[c + "_std" for c in cols])
rob = pd.DataFrame(RobustScaler().fit_transform(df[cols]), columns=[c + "_robust" for c in cols])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(df["Amount"], bins=60); ax[0].set_title("Amount (raw)")
ax[1].hist(std["Amount_std"], bins=60); ax[1].set_title("Amount (StandardScaler)")
ax[2].hist(rob["Amount_robust"], bins=60); ax[2].set_title("Amount (RobustScaler)")
plt.tight_layout()
savefig("01_scaling_comparison")
plt.show()

`V1-V28` are already scaled PCA outputs; `Time` and `Amount` are not, so they are scaled. **RobustScaler** (median and IQR) is chosen because `Amount` is heavily skewed with outliers.

**Important:** the cleaned file below is saved *unscaled*. The final scaler is fitted on the **training set only** in notebook 04, so no test information leaks into training.

## 7. Feature engineering: hour of day

In [ ]:
df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)
df[["Time", "Hour", "Amount", "Class"]].head()

## 8. Save cleaned data

In [ ]:
df.to_csv(f"{PROC_DIR}/creditcard_clean.csv", index=False)
print("Saved:", df.shape)
print(df["Class"].value_counts())

## Summary
- Rows after cleaning: see output above (duplicates removed), 0 missing values.
- Outliers kept deliberately; RobustScaler chosen for `Time` and `Amount` (applied after the split in notebook 04).
- New feature: `Hour`.
- Next: `02_eda.ipynb`.